In [ ]:
import os
import string
from collections import Counter
import pandas as pd
from deep_translator import GoogleTranslator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,  cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
import nltk
from nltk.corpus import stopwords
from imblearn.ensemble import BalancedRandomForestClassifier
import pickle

# DEFINE A FUNÇÃO PARA TRADUÇÃO DOS TEXTOS

In [2]:
def dividir_texto(texto, tamanho_maximo=4000):
    """Divide um texto grande em partes menores, substituindo quebras de linha por espaços"""
    # Substitui quebras de linha por espaços
    texto = texto.replace("\n", " ")
    
    # Divide o texto em partes menores
    return [texto[i:i+tamanho_maximo] for i in range(0, len(texto), tamanho_maximo)]

In [3]:
def tradutor(texto, idioma_destino):
    # Divide o texto grande em partes menores
    partes = dividir_texto(texto)

    # Lista para armazenar as partes traduzidas
    texto_traduzido = []
    
    # Traduzir cada parte
    for parte in partes:
        try:
            # Traduz cada parte do texto
            traduzido = GoogleTranslator(source='auto', target=idioma_destino).translate(parte)
            texto_traduzido.append(traduzido)
        except Exception as e:
            print(f"Erro ao traduzir parte: {e}")
            texto_traduzido.append(parte)  # Se houver erro, adiciona a parte original

    # Junta todas as partes traduzidas em um único texto
    return ' '.join(texto_traduzido)

# LÊ E MANIPULA O CATÁLOGO

In [ ]:
catalogo = pd.read_excel('./Discursos de Posse/_CATALOGO.xlsx', sheet_name='CATALOGO')

In [5]:
catalogo = catalogo.dropna(subset=['NOME_ARQUIVO_ORIGINAL'], ignore_index=True)

In [6]:
catalogo = catalogo.query('FLAG_POPULISMO_FINAL in (0, 1)').reset_index(drop=True)

# TREINA O MODELO

In [15]:
textos = []

In [ ]:
for nome_arquivo, lingua_original in zip(catalogo['NOME_ARQUIVO_ORIGINAL'], catalogo['LINGUA_ORIGINAL']):

    caminho = os.path.join('./Discursos de Posse/', nome_arquivo)

    with open(caminho, 'r', encoding='utf-8') as f:
        if lingua_original != 'ESPANHOL':
            textos.append(tradutor(f.read(), idioma_destino='es'))
        else:
            textos.append(f.read().replace("\n", " "))

In [17]:
catalogo['texto'] = textos

In [18]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\andre\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
stopwords_espanhol = stopwords.words('spanish')

In [27]:
vetorizador = TfidfVectorizer(max_features=100000, stop_words=stopwords_espanhol, ngram_range=(1, 2), min_df=2)

x = vetorizador.fit_transform(catalogo['texto'])
y = catalogo['FLAG_POPULISMO_FINAL']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=15)

modelo = BalancedRandomForestClassifier(n_estimators=3000, random_state=15, class_weight='balanced', max_depth=None, min_samples_leaf=2, sampling_strategy="all", n_jobs=8)

modelo.fit(x_train, y_train)

y_pred = modelo.predict(x_test)

# SALVA O MODELO

In [ ]:
pickle.dump(modelo, open('modelo.sav', 'wb'))

pickle.dump(vetorizador, open('vetorizador.sav', 'wb'))

In [ ]:
modelo = pickle.load(open('modelo.sav', 'rb'))

vetorizador = pickle.load(open('vetorizador.sav', 'rb'))

# TESTES DE PRECISÃO

In [28]:
print("Acurácia do Modelo:", accuracy_score(y_test, y_pred))
print(f"F1-Score médio: {cross_val_score(modelo, x, y, cv=5, scoring='f1_macro').mean()}")
print("Relatório de Classificação:\n", classification_report(y_test, y_pred))
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))

Acurácia do Modelo: 0.875
F1-Score médio: 0.7437190002979477
Relatório de Classificação:
               precision    recall  f1-score   support

           0       1.00      0.87      0.93        15
           1       0.33      1.00      0.50         1

    accuracy                           0.88        16
   macro avg       0.67      0.93      0.71        16
weighted avg       0.96      0.88      0.90        16

Matriz de Confusão:
 [[13  2]
 [ 0  1]]


# TOP 10 PALAVRAS POR IMPORTÂNCIA

In [29]:
feature_names = vetorizador.get_feature_names_out()
importances = modelo.feature_importances_
top_features = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
top_features = top_features.sort_values(by='Importance', ascending=False)
print(top_features.head(10))

             Feature  Importance
16295        quieren    0.005993
19609  transparencia    0.005610
19965             va    0.004985
18130             si    0.004745
7330       esfuerzos    0.004657
13448  oportunidades    0.004602
9292         haremos    0.004076
11962        mejores    0.004062
1225         américa    0.004009
636           además    0.003996


# DEFINE A FUNÇÃO PARA CHECAR O SCORE DE POPULISMO DO DISCURSO

In [30]:
def score_populismo(texto, modelo=modelo, vetorizador=vetorizador):
    texto_traduzido = tradutor(texto, idioma_destino='es')
    texto_vetor = vetorizador.transform([texto_traduzido])
    predicao = modelo.predict_proba(texto_vetor)
    return round(predicao[:, 1][0] * 100, 2)

In [ ]:
with open('./Discursos de Posse/BOL_EVO_MORALES_2.txt', 'r', encoding='utf-8') as f:
    a = f.read()

In [32]:
print(score_populismo(a))

88.54


# GERA UM DF COM O MODELO SENDO RODADO PARA TODOS OS DISCURSO

In [34]:
from tqdm.notebook import tqdm
from tqdm import tqdm

ranking_discurso = []

for arquivo, nome_lider, ano_mandato, nome_pais in tqdm(zip(catalogo['NOME_ARQUIVO_ORIGINAL'], 
                                                            catalogo['NOME_LIDER'], 
                                                            catalogo['PRIMEIRO_ANO_MANDATO'], 
                                                            catalogo['NOME_PAIS']), 
                                                        total=len(catalogo), desc="Processando discursos"):
    with open(f'C:/Users/andre/OneDrive/Faculdade/Mestrado/Dissertação/Código/Discursos de Posse/{arquivo}', 'r', encoding='utf-8') as f:
        a = f.read()

    score = score_populismo(a)

    ranking_discurso.append({'NOME_PAIS': nome_pais, 'NOME_LIDER': nome_lider, 'ANO_MANDATO': ano_mandato, 'SCORE': score})

df_ranking = pd.DataFrame(ranking_discurso)

Processando discursos: 100%|██████████| 79/79 [05:30<00:00,  4.19s/it]


In [35]:
ranking = df_ranking.sort_values(by='SCORE', ascending=False, ignore_index=True)

In [36]:
ranking

,NOME_PAIS,NOME_LIDER,ANO_MANDATO,SCORE
0,Bolivia,Juan Evo Morales Ayma,2010,88.54
1,Venezuela,Nicolás Maduro,2013,87.34
2,Venezuela,Nicolás Maduro,2019,86.95
3,Bolivia,Juan Evo Morales Ayma,2006,85.86
4,Ecuador,Rafael Vicente Correa Delgado,2013,85.46
...,...,...,...,...
74,Mexico,Vicente Fox Quesada,2000,25.83
75,Costa Rica,Carlos Alvarado Quesada,2018,25.55
76,Peru,Francisco Rafael Sagasti Hochhausler,2020,25.51
77,Colombia,Iván Duque Márquez,2018,24.78


In [ ]:
ranking.to_excel('ranking.xlsx', index=False)

# CONTADOR DE PALAVRAS

In [ ]:
diretorio = './Discursos de Posse'

contador_total = Counter()

for arquivo in os.listdir(diretorio):
    if arquivo.endswith('.txt'):
        caminho_arquivo = os.path.join(diretorio, arquivo)
        
        with open(caminho_arquivo, 'r', encoding='utf-8') as f:
            texto = f.read()
        
        # Normalização do texto
        texto = texto.lower()  # Converte para minúsculas
        texto = texto.translate(str.maketrans('', '', string.punctuation))  # Remove pontuação
        
        # Divide o texto em palavras e conta
        palavras = texto.split()
        contador_total.update(palavras)

df_palavras = pd.DataFrame(contador_total.items(), columns=['Palavra', 'Frequência']).sort_values(by='Frequência', ascending=False, ignore_index=True)

df_palavras


,Palavra,Frequência
0,de,22816
1,la,14652
2,y,12941
3,que,12138
4,el,9261
...,...,...
27078,llueva,1
27079,truena,1
27080,quio,1
27081,ves,1
